# SayIt: Validasi Silang k-fold Deteksi Uang Rupiah

Notebook ini menjawab satu pertanyaan yang notebook 02 tidak bisa jawab:
**bagaimana kalau 35 foto sendiri dipakai untuk MELATIH, bukan hanya untuk menguji?**

Di notebook 02 seluruh foto `sendiri=True` menjadi split test. Itu memberi angka yang
jujur, tetapi berarti foto paling berharga yang kita punya - foto uang sungguhan dari
kamera yang akan dipakai - **tidak pernah dilatihkan sama sekali**.

k-fold memberi dua-duanya:

1. Foto sendiri dibagi `K` kelompok. Untuk tiap putaran, satu kelompok menjadi test dan
   sisanya ikut data latih. Dilatih `K` kali.
2. Tiap foto sendiri diuji **tepat sekali**, oleh model yang belum pernah melihatnya.
   Jadi angka gabungannya tetap jujur, tetapi tiap foto juga ikut melatih `K-1` model.
3. Setelah itu satu model terakhir dilatih pada **seluruh** data untuk dipakai aplikasi.

**Keterbatasan yang wajib disebut di laporan:** angka k-fold mengukur *prosedurnya*,
bukan model terakhir. Setelah model terakhir dilatih pada semua foto sendiri, tidak ada
lagi data tersisa untuk mengujinya - akurasi sebenarnya hanya bisa diukur dengan foto
BARU yang belum pernah ada di notebook ini.

Pembanding yang harus dikalahkan: **65,7%** (metrik total, 23 benar / 6 salah / 6 diam
dari 35 foto), diukur 06-10 di laptop dengan `scripts/diagnosa_uang.py`.

## 1. Persiapan

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics roboflow imagehash

In [ ]:
import hashlib, json, re, random, shutil
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image as IPImage, display
from PIL import Image
from roboflow import Roboflow
from ultralytics import YOLO

DRIVE_DIR = Path('/content/drive/MyDrive/sayit')
RUNS_DIR = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
DATA_DIR = DRIVE_DIR / 'datasets'
for d in (RUNS_DIR, MODELS_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Konfigurasi

In [ ]:
# ====== ISI BAGIAN INI ======
# Dataset dari Roboflow. Tambah sebanyak yang perlu, urutan tidak penting.
#   mode 'label_dari_nama' : label hanya dipakai bila nominal tertulis di nama berkas, dan label
#                            diganti mengikuti nama itu. Untuk dataset A yang sebagian labelnya tertukar.
#   mode 'percaya_label'   : label dipakai apa adanya, tetapi gambar dibuang bila labelnya
#                            bertentangan dengan nominal di nama berkas.
#   sendiri=True           : foto tim sendiri. SELURUHNYA masuk split test, tidak pernah ikut latih.
DATASETS = [
    dict(nama='A', workspace='mochammadalthofsr-gmail-com',
         project='deteksi-rupiah-jkkmu-zagcv', versi=1, mode='label_dari_nama'),
    dict(nama='B', workspace='mochammadalthofsr-gmail-com',
         project='deteksi-mata-uang-rupiah-nerog-ltccw-5oxma-tz6tz', versi=1, mode='percaya_label'),

    # Tambahan dataset publik. Pilih versi TANPA augmentasi, dan pastikan nama kelas di Roboflow
    # sudah persis '1000' ... '100000' (sel unduh berhenti kalau ada nama lain).
    # dict(nama='D', workspace='...', project='...', versi=1, mode='percaya_label'),
    # dict(nama='E', workspace='...', project='...', versi=1, mode='percaya_label'),

    # Foto sendiri yang sudah dilabeli. Selama baris ini masih dikomentari, split test memakai
    # data publik dan angkanya TIDAK mewakili dunia nyata.
    # Dua cara, pilih salah satu:
    #   a) folder di Google Drive (hasil scripts/buat_label_uang.py, tinggal disalin ke Drive):
    dict(nama='SENDIRI', lokal='/content/drive/MyDrive/sayit/data/label_sendiri',
         mode='percaya_label', sendiri=True),
    #   b) diunggah ke Roboflow seperti dataset lain:
    # dict(nama='SENDIRI', workspace='...', project='...', versi=1, mode='percaya_label', sendiri=True),
]

CLASSES = ['1000', '2000', '5000', '10000', '20000', '50000', '100000']

SPLIT_RATIO = (0.7, 0.2, 0.1)   # train, val, test. Bagian test diabaikan bila ada dataset sendiri=True
SEED = 42
PHASH_MAX_DIST = 8              # jarak Hamming maksimum agar dua foto dianggap "mirip"

MODEL_WEIGHTS = 'yolov8n.pt'
EPOCHS   = 100
PATIENCE = 20
IMGSZ    = 640
BATCH    = 16
RUN_NAME = 'money_v2_3'         # ganti tiap percobaan agar run sebelumnya tidak tertimpa
CONF_THRESHOLD = 0.5

# Augmentasi. Yang tidak disebut di sini memakai bawaan ultralytics.
# Alasan tiap angka ada di sel penjelasan sebelum training.
AUG = dict(
    fliplr=0.0,          # uang cermin tidak pernah ada di dunia nyata
    degrees=180,         # WAJIB: pengguna tunanetra tidak bisa membidik kamera
    perspective=0.0005,  # difoto dari sudut, bukan tegak lurus di atas meja
    scale=0.6,           # jarak kamera ke uang sangat bervariasi
    translate=0.2,       # uang tidak selalu di tengah bingkai
    hsv_v=0.5,           # lampu kuning, siang terang, bayangan tangan
    hsv_s=0.8,
)

# Anotasi bagian uang (potret, angka, benang pengaman) yang berada di dalam kotak uang utama dibuang
NESTED_CONTAIN = 0.85           # kotak dibuang jika >= 85% luasnya berada di dalam kotak lain berkelas sama

# Contoh latar tanpa uang dari COCO128 untuk mengurangi deteksi palsu
ADD_COCO_BACKGROUNDS = True
N_BACKGROUNDS = 120
COCO128_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip'

# Model pembanding pada split test yang sama.
BASELINE_PT = RUNS_DIR / 'money_v2_2' / 'weights' / 'best.pt'
# ============================

random.seed(SEED)
np.random.seed(SEED)

def ambil_kunci_roboflow() -> str:
    """Ambil API key Roboflow, tahan terhadap perbedaan versi Colab.

    Sebagian versi `google.colab.userdata.get()` mengembalikan str, sebagian
    mengembalikan dict. Kalau keduanya gagal, kunci diminta lewat ketikan.
    """
    kunci = None
    try:
        from google.colab import userdata
        kunci = userdata.get('ROBOFLOW_API_KEY')
    except Exception as e:
        print('Secret ROBOFLOW_API_KEY tidak terbaca:', type(e).__name__)
    if isinstance(kunci, dict):
        kunci = (kunci.get('ROBOFLOW_API_KEY') or kunci.get('value')
                 or next((v for v in kunci.values() if isinstance(v, str)), None))
    if not isinstance(kunci, str) or not kunci.strip():
        from getpass import getpass
        kunci = getpass('Tempel Roboflow API key (Settings > API Keys di roboflow.com): ')
    return kunci.strip()


RF_KEY = ambil_kunci_roboflow()
assert RF_KEY, 'API key kosong.'
rf = Roboflow(api_key=RF_KEY)


## 3. Siapkan semua dataset

Tiap entri `DATASETS` boleh berasal dari **Roboflow** (`workspace` + `project` + `versi`) atau dari
**folder lokal di Google Drive** (`lokal='/content/drive/MyDrive/...'`). Drive sudah ter-mount di
Bagian 1, jadi folder apa pun di sana bisa dibaca langsung tanpa lewat Roboflow.

Folder lokal harus berbentuk dataset YOLO: `<root>/test/images/*.jpg` dan `<root>/test/labels/*.txt`,
dengan `data.yaml` berisi daftar nama kelas. Bentuk itu persis yang dihasilkan
`scripts/buat_label_uang.py` di laptop, jadi tinggal salin foldernya ke Drive.

Sel ini berhenti keras bila ada nama kelas yang tidak dikenal. Itu disengaja: nama kelas yang
berbeda antar-dataset (`Rp1000`, `1k`, `1000 rupiah`) adalah sumber salah label yang paling mudah
lolos tanpa disadari. Rapikan namanya menjadi `1000` sampai `100000`, lalu ulangi.


In [ ]:
def dari_roboflow(d):
    ds = rf.workspace(d['workspace']).project(d['project']).version(d['versi']).download(
        'yolov8', location=f"/content/src_{d['nama']}", overwrite=True)
    return Path(ds.location)


def baca_names(root: Path):
    """Daftar nama kelas dari data.yaml. Tanpa data.yaml, urutan CLASSES yang dipakai."""
    yml = root / 'data.yaml'
    if not yml.exists():
        print(f'  (tanpa data.yaml, dianggap memakai urutan CLASSES)')
        return list(CLASSES)
    with open(yml) as f:
        names = yaml.safe_load(f)['names']
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    return [str(n) for n in names]


def siapkan(d):
    """Root dan nama kelas satu dataset, dari Roboflow atau dari folder Drive."""
    if d.get('lokal'):
        root = Path(d['lokal'])
        assert root.exists(), f"{d['nama']}: folder tidak ada: {root}"
        asal = f'lokal {root}'
    else:
        root = dari_roboflow(d)
        asal = f"roboflow {d['project']}"
    names = baca_names(root)
    unknown = set(names) - set(CLASSES)
    assert not unknown, f"{d['nama']}: kelas tidak dikenal {unknown}. Rapikan nama kelas dulu."
    # dihitung di sini, bukan lewat image_files() dari Bagian 4, supaya sel ini tidak
    # bergantung pada urutan menjalankan dan Runtime > Run all tetap aman
    n_gambar = sum(1 for split in ('train', 'valid', 'val', 'test')
                   for p in (root / split / 'images').glob('*') if p.is_file())
    assert n_gambar, f"{d['nama']}: tidak ada gambar di {root}. Cek strukturnya: <root>/test/images/"
    print(f"{d['nama']:10s} {asal:55s} {n_gambar:>5d} gambar, kelas {names}")
    return root, names


for d in DATASETS:
    d['root'], d['names'] = siapkan(d)

punya_sendiri = [d['nama'] for d in DATASETS if d.get('sendiri')]
print()
if punya_sendiri:
    print('Dataset foto sendiri:', ', '.join(punya_sendiri), '-> seluruhnya menjadi split test.')
else:
    print('PERINGATAN: belum ada dataset foto sendiri (sendiri=True).')
    print('            Split test akan berasal dari sumber yang sama dengan data latih, sehingga')
    print('            mAP di Bagian 10 optimistis dan tidak mewakili dunia nyata.')
    print('            Ukur yang sebenarnya di laptop:')
    print(r'              python scripts\diagnosa_uang.py data\uji_sendiri --harapan-dari-nama')


## 4. Fungsi bantu

Membaca label (poligon diubah menjadi kotak, anotasi ganda dibuang), membaca nominal dari nama file, dan menentukan foto asal.


In [ ]:
K_RE = re.compile(r'(?i)(?<!\d)(1|2|5|10|20|50|100)\s*k{1,2}(?![a-z])')
N_RE = re.compile(r'(?<!\d)(1000|2000|5000|10000|20000|50000|100000)(?!\d)')

def nominal_from_name(name):
    m = K_RE.search(name)
    if m:
        return str(int(m.group(1)) * 1000)
    m = N_RE.search(name)
    return m.group(1) if m else None

def source_name(img: Path):
    return img.name.split('.rf.')[0]

def xywh_to_xyxy(a):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2,
                     a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], axis=1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def read_labels(lbl_path: Path, names, dup_iou=0.7):
    """Kembalikan list (nama_kelas, [xc, yc, w, h]) tanpa duplikat."""
    items = []
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            vals = list(map(float, parts[1:]))
            if len(vals) != 4:
                xs, ys = vals[0::2], vals[1::2]
                vals = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
            items.append((names[int(parts[0])], vals))
    kept = []
    for c, box in items:
        bx = xywh_to_xyxy(box)
        if not any(c == kc and iou_matrix(bx, xywh_to_xyxy(kb))[0, 0] > dup_iou for kc, kb in kept):
            kept.append((c, box))
    return kept

def nested_mask(classes, xyxy, contain_thr=0.85):
    """True untuk kotak yang >= contain_thr luasnya berada di dalam kotak LAIN yang lebih besar dan berkelas sama."""
    xyxy = np.asarray(xyxy, dtype=float).reshape(-1, 4)
    area = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    mask = np.zeros(len(xyxy), dtype=bool)
    for i in range(len(xyxy)):
        for j in range(len(xyxy)):
            if i == j or classes[i] != classes[j] or area[j] <= area[i]:
                continue
            iw = min(xyxy[i, 2], xyxy[j, 2]) - max(xyxy[i, 0], xyxy[j, 0])
            ih = min(xyxy[i, 3], xyxy[j, 3]) - max(xyxy[i, 1], xyxy[j, 1])
            if iw > 0 and ih > 0 and iw * ih / (area[i] + 1e-9) >= contain_thr:
                mask[i] = True
                break
    return mask

def drop_nested(items, contain_thr=0.85):
    if len(items) < 2:
        return items, 0
    mask = nested_mask([c for c, _ in items], xywh_to_xyxy([b for _, b in items]), contain_thr)
    return [it for it, m in zip(items, mask) if not m], int(mask.sum())

def image_files(root: Path):
    for split in ('train', 'valid', 'val', 'test'):
        d = root / split / 'images'
        if d.exists():
            for p in sorted(d.iterdir()):
                if p.is_file():
                    yield p

def black_fraction(p: Path):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return float((im < 10).mean()) if im is not None else 1.0

def phash_bits(p: Path):
    with Image.open(p) as im:
        return imagehash.phash(im.convert('RGB')).hash.flatten()

## 5. Seleksi data

In [ ]:
records = []
log = Counter()
nested_examples = []
dilewati = []


def tambah_percaya_label(d):
    """Label dipakai apa adanya, kecuali bertentangan dengan nominal di nama berkas."""
    for img in image_files(d['root']):
        labels = read_labels(img.parent.parent / 'labels' / f'{img.stem}.txt', d['names'])
        labels, n_nested = drop_nested(labels, NESTED_CONTAIN)
        if n_nested:
            log[f"{d['nama']} kotak bersarang dibuang"] += n_nested
            nested_examples.append((img, d['names']))
        src = source_name(img)
        nom = nominal_from_name(src)
        if nom and len(labels) == 1 and labels[0][0] != nom:
            log[f"{d['nama']} dibuang: nama != label"] += 1
            continue
        # Foto SENDIRI tanpa anotasi BUKAN gambar latar: ia foto uang yang belum
        # dilabeli. Label kosong di YOLO berarti "tidak ada uang di gambar ini",
        # jadi membiarkannya mengajari model KEBALIKAN dari yang benar - dan bila
        # ia jatuh ke split test, ia tidak punya acuan untuk dinilai sama sekali.
        # Dataset publik sengaja TIDAK dikecualikan di sini: di sana gambar tanpa
        # objek memang ada dengan sengaja sebagai contoh latar.
        if d.get('sendiri') and not labels:
            log[f"{d['nama']} DILEWATI: foto sendiri tanpa label"] += 1
            dilewati.append(f"{d['nama']}/{img.name}")
            continue
        records.append({'dataset': d['nama'], 'path': img, 'asal': f"{d['nama']}/{src}",
                        'labels': labels, 'sendiri': bool(d.get('sendiri'))})
        log[f"{d['nama']} dipakai"] += 1


def tambah_label_dari_nama(d):
    """Hanya gambar satu objek yang nominalnya tertulis di nama berkas, plus gambar tanpa objek.

    Label asli diabaikan dan diganti sesuai nama berkas, karena dataset ini terbukti punya label
    tertukar. Satu salinan per foto asal: dipilih yang bingkai hitamnya paling sedikit, sehingga
    hasil rotasi Roboflow tidak ikut terbawa.
    """
    kandidat = defaultdict(list)
    for img in image_files(d['root']):
        labels = read_labels(img.parent.parent / 'labels' / f'{img.stem}.txt', d['names'])
        src = source_name(img)
        nom = nominal_from_name(src)
        if len(labels) == 0:
            kandidat[src].append((img, []))
        elif nom and len(labels) == 1:
            kandidat[src].append((img, [(nom, labels[0][1])]))
        else:
            log[f"{d['nama']} dibuang: tidak bisa diverifikasi"] += 1
    for src, copies in kandidat.items():
        img, labels = min(copies, key=lambda t: black_fraction(t[0]))
        # penjaga yang sama seperti di tambah_percaya_label, lihat alasannya di sana
        if d.get('sendiri') and not labels:
            log[f"{d['nama']} DILEWATI: foto sendiri tanpa label"] += 1
            dilewati.append(f"{d['nama']}/{img.name}")
            continue
        records.append({'dataset': d['nama'], 'path': img, 'asal': f"{d['nama']}/{src}",
                        'labels': labels, 'sendiri': bool(d.get('sendiri'))})
        log[f"{d['nama']} dipakai (latar)" if not labels
            else f"{d['nama']} dipakai (label dari nama file)"] += 1
        log[f"{d['nama']} salinan dibuang"] += len(copies) - 1


CARA = {'percaya_label': tambah_percaya_label, 'label_dari_nama': tambah_label_dari_nama}
for d in DATASETS:
    assert d['mode'] in CARA, f"mode tidak dikenal: {d['mode']}"
    CARA[d['mode']](d)

for k, v in sorted(log.items()):
    print(f'{k:48s}: {v}')
print(f'Total gambar: {len(records)}')
print('Per dataset :', dict(Counter(r['dataset'] for r in records)))

if dilewati:
    print()
    print(f'{len(dilewati)} foto SENDIRI dilewati karena tidak punya label terisi:')
    for nama in dilewati:
        print('   ', nama)
    print('Ini DISENGAJA. Berkas aslinya di Drive tidak disentuh.')
    print('Labeli foto itu lalu jalankan ulang bila ingin dipakai.')

n_sendiri_rec = sum(1 for r in records if r['sendiri'])
assert n_sendiri_rec, 'Tidak ada satu pun foto sendiri berlabel yang masuk records.'
assert all(r['labels'] for r in records if r['sendiri']), (
    'Masih ada foto sendiri tanpa label di records - penjaga di atas tidak jalan.')
print(f'Foto sendiri berlabel yang dipakai: {n_sendiri_rec}')


### 5.1 Cek gambar yang kotak bersarangnya dibuang

Beberapa foto di dataset B memberi kotak terpisah untuk bagian uang (potret, angka nominal, benang pengaman). Model jadi belajar menghitung satu lembar sebagai beberapa lembar. Gambar di bawah menampilkan **kotak yang tersisa** setelah penyaringan. Pastikan setiap uang tetap punya satu kotak, dan tidak ada uang asli yang ikut hilang.

In [ ]:
show = nested_examples[:12]
print(f'{len(nested_examples)} gambar terdampak, ditampilkan {len(show)}')
if show:
    fig, axes = plt.subplots((len(show) + 3) // 4, 4, figsize=(16, 4 * ((len(show) + 3) // 4)))
    axes = np.atleast_1d(axes).flatten()
    for ax, (p, names) in zip(axes, show):
        items, _ = drop_nested(read_labels(p.parent.parent / 'labels' / f'{p.stem}.txt', names), NESTED_CONTAIN)
        im = cv2.imread(str(p)); h, w = im.shape[:2]
        for c, (xc, yc, bw, bh) in items:
            x1, y1, x2, y2 = int((xc - bw / 2) * w), int((yc - bh / 2) * h), int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), 3)
            cv2.putText(im, c, (x1 + 4, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)
        ax.imshow(im[:, :, ::-1]); ax.set_title(p.name[:30], fontsize=8); ax.axis('off')
    for ax in axes[len(show):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

### 5.2 Tambah contoh latar tanpa uang (COCO128)

Dataset B tidak punya foto tanpa uang, sehingga model mudah mendeteksi uang palsu pada kain merah, kardus, dan sejenisnya. Sel ini menambahkan gambar acak dari COCO128 (foto umum tanpa uang) sebagai contoh latar dengan label kosong.

In [ ]:
import urllib.request, zipfile
if ADD_COCO_BACKGROUNDS:
    try:
        zpath = Path('/content/coco128.zip')
        if not zpath.exists():
            urllib.request.urlretrieve(COCO128_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall('/content')
        coco_imgs = sorted(Path('/content/coco128/images/train2017').glob('*.jpg'))
        picked = random.Random(SEED).sample(coco_imgs, k=min(N_BACKGROUNDS, len(coco_imgs)))
        for p in picked:
            records.append({'dataset': 'LATAR', 'path': p, 'asal': 'LATAR/' + p.stem,
                            'labels': [], 'sendiri': False})
        print(f'{len(picked)} gambar latar COCO128 ditambahkan. Total gambar v2: {len(records)}')
    except Exception as e:
        print('Gagal mengunduh COCO128, lanjut tanpa latar tambahan:', e)

## 6. Kelompokkan foto yang mirip

Foto berurutan dari sesi pemotretan yang sama bisa sangat mirip. Foto seperti itu dikelompokkan agar tidak terpisah ke train dan test. Jika ada kelompok yang terlalu besar, ambang kemiripan diturunkan otomatis.

In [ ]:
H = np.stack([phash_bits(r['path']) for r in records])
N = len(records)

def group_with(max_dist):
    parent = list(range(N))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(0, N, 200):
        d = (H[i:i + 200, None, :] != H[None, :, :]).sum(-1)
        for a, b in zip(*np.nonzero(d <= max_dist)):
            a += i
            if a < b:
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[ra] = rb
    return [find(i) for i in range(N)]

dist = PHASH_MAX_DIST
while True:
    groups = group_with(dist)
    sizes = Counter(groups)
    largest = max(sizes.values())
    print(f'jarak <= {dist}: {len(sizes)} kelompok, kelompok terbesar {largest} gambar')
    if largest <= max(10, 0.03 * N) or dist <= 2:
        break
    dist -= 2

for r, g in zip(records, groups):
    r['grup'] = g
print('Sebaran ukuran kelompok:', dict(sorted(Counter(sizes.values()).items())))

## 7. Konfigurasi k-fold

`EPOCHS_FOLD` sengaja dibuat terpisah dan lebih kecil daripada `EPOCHS_FINAL`: putaran
k-fold hanya untuk mengukur, jadi tidak perlu dilatih selama model yang dipakai.
Dengan `K=5` dan `EPOCHS_FOLD=60`, total latihan sekitar 5 x 60 + 100 epoch.
Perkirakan waktunya di T4 sebelum menjalankan semuanya.

In [ ]:
K = 5                    # jumlah putaran; 5 berarti tiap putaran menguji sekitar 1/5 foto sendiri
EPOCHS_FOLD  = 60        # untuk putaran pengukuran
EPOCHS_FINAL = 100       # untuk model yang dipakai aplikasi
# v2 sejak 23-10. JANGAN memakai ulang nama run lama: sel 10 melewati latihan
# bila bobotnya sudah ada di Drive, sehingga bobot v1 (dilatih pada 48 foto) akan
# dinilai diam-diam memakai split test v2 (80 foto). Sel 10 sekarang juga menyimpan
# sidik jari isi lipatan dan BERHENTI KERAS bila bobot lama tidak cocok datanya.
RUN_PREFIX = 'money_kfold_v3'   # v3 = dataset 80 foto, sesudah anotasi cacat dikeluarkan 09-11.
                                # JANGAN hapus folder money_kfold_v2_* di Drive: itu bukti angka 90,1%
                                # yang dikutip jurnal, dan satu-satunya pembanding untuk v3.

# Ambang dan penyaringan dibuat SAMA dengan yang dipakai aplikasi, supaya angka di sini
# bisa dibandingkan langsung dengan hasil scripts/diagnosa_uang.py.
#   config/settings.yaml -> vision.money.score_min = 0.30 (diturunkan 22-10)
CONF_EVAL = 0.30
# Aplikasi juga membuang deteksi yang skornya jauh di bawah deteksi TERKUAT di foto
# yang sama (sayit/vision/money.py -> filter_by_relative_score). Tanpa ini angka
# notebook lebih PESIMIS daripada yang dialami pengguna: kesalahan "1 lembar lawan 2"
# justru pola yang dirancang dibuang aturan tersebut.
#   config/settings.yaml -> vision.money.relative_score_min = 0.80
REL_EVAL = 0.80
print(f'{K} putaran, {EPOCHS_FOLD} epoch per putaran, lalu {EPOCHS_FINAL} epoch untuk model terakhir')
print(f'Ambang evaluasi {CONF_EVAL} (sama dengan vision.money.score_min di aplikasi)')

## 8. Bagi foto sendiri menjadi K lipatan

Pembagian dilakukan atas **kelompok kemiripan** (`grup`), bukan atas berkas, supaya dua
foto yang nyaris identik tidak pernah terpisah antara latih dan uji. Itu aturan yang sama
dengan notebook 02, dan alasannya sama: tanpa itu angkanya bocor dan terlalu bagus.

Lipatan diseimbangkan per nominal, agar tiap putaran menguji campuran kelas yang mirip.

In [ ]:
from collections import Counter, defaultdict
import random

def main_class(rs):
    c = Counter(lbl for r in rs for lbl, _ in r['labels'])
    return c.most_common(1)[0][0] if c else 'latar'

by_group = defaultdict(list)
for r in records:
    by_group[r['grup']].append(r)

grup_sendiri = sorted(g for g, rs in by_group.items() if any(r['sendiri'] for r in rs))
grup_publik  = sorted(g for g in by_group if g not in set(grup_sendiri))

assert grup_sendiri, (
    'Tidak ada dataset dengan sendiri=True. k-fold tidak ada gunanya tanpa foto sendiri: '
    'aktifkan entri SENDIRI di DATASETS pada Bagian 2.')

# lipatan diseimbangkan per nominal dominan
per_kelas = defaultdict(list)
for g in grup_sendiri:
    per_kelas[main_class(by_group[g])].append(g)

rng = random.Random(SEED)
fold_of_group = {}
# Penghitung berjalan MELINTASI kelas, tidak dimulai ulang per kelas. Kalau dimulai
# ulang, kelas yang hanya punya 2 kelompok selalu mengisi lipatan 0 dan 1, sehingga
# lipatan terakhir bisa kosong sama sekali. Terbukti lewat uji lokal: 12 kelompok
# atas 7 kelas dengan K=4 membuat lipatan 2 dan 3 kosong.
berikutnya = 0
for cls, gs in sorted(per_kelas.items()):
    rng.shuffle(gs)
    for g in gs:
        fold_of_group[g] = berikutnya % K
        berikutnya += 1

ringkas = []
for k in range(K):
    gs = [g for g, f in fold_of_group.items() if f == k]
    foto = [r for g in gs for r in by_group[g]]
    cnt = Counter(lbl for r in foto for lbl, _ in r['labels'])
    ringkas.append({'fold': k, 'kelompok': len(gs), 'foto': len(foto),
                    **{c: cnt.get(c, 0) for c in CLASSES}})
fold_stats = pd.DataFrame(ringkas).set_index('fold')
display(fold_stats)

n_sendiri = sum(len(by_group[g]) for g in grup_sendiri)
print(f'{n_sendiri} foto sendiri dalam {len(grup_sendiri)} kelompok, dibagi {K} lipatan.')
print(f'{sum(len(by_group[g]) for g in grup_publik)} foto publik selalu menjadi latih/validasi.')
kosong = [k for k in range(K) if not any(f == k for f in fold_of_group.values())]
assert not kosong, f'Lipatan {kosong} kosong. Turunkan K (sekarang {K}).'

## 9. Fungsi bantu: tulis dataset satu putaran, lalu evaluasi

Dipisah menjadi fungsi supaya perulangan di bawah tetap pendek dan supaya kedua langkah
ini bisa diuji sendiri. `tulis_dataset` memakai konvensi nama yang sama dengan notebook 02.

In [ ]:
def tulis_dataset(split_of_record, root):
    """Tulis dataset YOLO ke `root`. `split_of_record` memetakan id(record) -> split."""
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    for split in ('train', 'val', 'test'):
        (root / split / 'images').mkdir(parents=True)
        (root / split / 'labels').mkdir(parents=True)

    for i, r in enumerate(records):
        split = split_of_record[id(r)]
        stem = f"{r['dataset']}_{i:05d}"
        shutil.copy(r['path'], root / split / 'images' / f"{stem}{r['path'].suffix}")
        lines = [f'{CLASSES.index(c)} ' + ' '.join(f'{v:.6f}' for v in box) for c, box in r['labels']]
        (root / split / 'labels' / f'{stem}.txt').write_text('\n'.join(lines))

    yml = root / 'data.yaml'
    with open(yml, 'w') as f:
        yaml.safe_dump({'path': str(root),
                        'train': str(root / 'train' / 'images'),
                        'val': str(root / 'val' / 'images'),
                        'test': str(root / 'test' / 'images'),
                        'names': {i: c for i, c in enumerate(CLASSES)}}, f, sort_keys=False)
    return yml


def evaluasi_total(model, root, conf=None, rel=None):
    """Metrik TOTAL: apakah jumlah rupiah yang DIUCAPKAN ke pengguna benar.

    Bukan metrik kelas dominan. Terbukti 06-10 bahwa keduanya berbeda: satu lembar
    100.000 yang terdeteksi dua kali tetap benar secara kelas, tetapi diucapkan
    "totalnya dua ratus ribu rupiah". Yang dialami pengguna adalah totalnya.
    """
    # conf bisa dioper supaya ambang dapat DISAPU tanpa melatih ulang:
    # bobot tiap lipatan sudah tersimpan di Drive, jadi menilai ulang itu murah.
    conf = CONF_EVAL if conf is None else conf
    rel = REL_EVAL if rel is None else rel
    root = Path(root)

    # Foto uji WAJIB punya acuan. Kalau tidak, ia tidak bisa masuk kategori mana pun,
    # dan membiarkannya lewat diam-diam adalah kegagalan paling berbahaya di skrip ukur.
    tanpa_acuan = [p.name for p in sorted((root / 'test' / 'images').iterdir())
                   if not (root / 'test' / 'labels' / f'{p.stem}.txt').exists()
                   or not (root / 'test' / 'labels' / f'{p.stem}.txt').read_text().strip()]
    assert not tanpa_acuan, (
        f'{len(tanpa_acuan)} foto uji tidak punya label terisi: ' + ', '.join(tanpa_acuan[:5])
        + '. Foto sendiri tanpa label seharusnya sudah dilewati di Bagian 5.')

    rows = []
    for img in sorted((root / 'test' / 'images').iterdir()):
        baris = (root / 'test' / 'labels' / f'{img.stem}.txt').read_text().splitlines()
        gt = [CLASSES[int(l.split()[0])] for l in baris if l.strip()]
        res = model.predict(source=str(img), conf=conf, imgsz=IMGSZ, verbose=False)[0]
        cls = res.boxes.cls.cpu().numpy().astype(int)
        kotak = res.boxes.xyxy.cpu().numpy()
        skor = res.boxes.conf.cpu().numpy()
        simpan = ~nested_mask(list(cls), kotak, NESTED_CONTAIN) if len(cls) else []
        sisa = [(c, s) for c, s, keep in zip(cls, skor, simpan) if keep]
        # Ambang relatif, URUTANNYA sesudah nested - sama persis dengan
        # sayit/vision/money.py. Dibalik, lembar utuh yang berskor lebih rendah
        # daripada potret di dalamnya justru yang terbuang.
        if rel > 0 and len(sisa) > 1:
            kuat = max(s for _, s in sisa)
            sisa = [(c, s) for c, s in sisa if s >= kuat * rel]
        pred = [CLASSES[c] for c, _ in sisa]
        rows.append({'file': img.name,
                     'total_benar': sum(map(int, gt)),
                     'total_prediksi': sum(map(int, pred)),
                     'lembar_benar': len(gt), 'lembar_prediksi': len(pred)})
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Foto TANPA label (total_benar = 0) tidak bisa dinilai, dan kalau dibiarkan
    # ia merusak metrik secara diam-diam: model yang juga tidak mendeteksi apa pun
    # menghasilkan 0 == 0, jadi foto itu dihitung BENAR sekaligus DIAM - masuk dua
    # kolom sekaligus. Terjadi 22-10: tiap baris sapuan berjumlah 52 dari 50 foto,
    # dan dua jawaban "benar" itu sebenarnya foto yang labelnya memang tidak ada.
    tanpa_label = df[df.total_benar == 0]
    if len(tanpa_label):
        print('    ' + str(len(tanpa_label)) + ' foto TANPA label dibuang dari metrik: '
              + ', '.join(tanpa_label.file))
        df = df[df.total_benar > 0].reset_index(drop=True)

    # Urutannya penting: diam dihitung dulu, lalu benar MENGECUALIKAN diam.
    df['diam'] = df.total_prediksi == 0
    df['benar'] = (df.total_benar == df.total_prediksi) & ~df.diam
    df['salah'] = ~df.benar & ~df.diam
    return df

## 10. Jalankan K putaran

Tiap putaran: foto sendiri di lipatan ini menjadi test, foto sendiri lain ikut latih,
foto publik dibagi latih/validasi seperti biasa.

Kalau sesi Colab terputus di tengah, jalankan ulang sel ini. Putaran yang bobotnya sudah
ada akan dilewati, jadi tidak ada latihan yang terbuang.

In [ ]:
rng_pub = random.Random(SEED)
pub = list(grup_publik)
rng_pub.shuffle(pub)
n_val = max(1, round(len(pub) * 0.2)) if len(pub) >= 3 else 0
grup_val = set(pub[:n_val])

def sidik_lipatan(split_of_record):
    """Sidik jari isi satu lipatan: berubah bila daftar foto ATAU pembagiannya berubah.

    Dipakai supaya bobot yang sudah ada hanya boleh dipakai ulang bila ia memang
    dilatih dari lipatan yang sama. Tanpa ini, menambah foto ke dataset lalu
    menjalankan ulang notebook akan MENILAI MODEL LAMA dengan data baru tanpa
    satu pun pesan - persis kesalahan yang sudah dua kali terjadi di proyek ini.
    """
    isi = sorted(f"{split_of_record[id(r)]}:{r['asal']}" for r in records)
    return hashlib.sha1(chr(10).join(isi).encode()).hexdigest()


hasil_fold = {}
for k in range(K):
    run = f'{RUN_PREFIX}_fold{k}'
    best = RUNS_DIR / run / 'weights' / 'best.pt'

    uji = {g for g, f in fold_of_group.items() if f == k}
    split_of_record = {}
    for g, rs in by_group.items():
        if g in uji:
            split = 'test'
        elif g in grup_val:
            split = 'val'
        else:
            split = 'train'
        for r in rs:
            split_of_record[id(r)] = split

    root = Path(f'/content/data_fold{k}')
    yml = tulis_dataset(split_of_record, root)
    n = Counter(split_of_record.values())
    print(f'--- fold {k}: latih {n["train"]}, validasi {n["val"]}, uji {n["test"]}', flush=True)

    # kebocoran diperiksa ulang tiap putaran, bukan sekali di awal
    per_grup = defaultdict(set)
    for g, rs in by_group.items():
        for r in rs:
            per_grup[g].add(split_of_record[id(r)])
    bocor = [g for g, s in per_grup.items() if len(s) > 1]
    assert not bocor, f'fold {k}: {len(bocor)} kelompok ada di lebih dari satu split'
    asing = [r for r in records if split_of_record[id(r)] == 'test' and not r['sendiri']]
    assert not asing, f'fold {k}: {len(asing)} foto bukan milik sendiri masuk split test'

    sidik_baru = sidik_lipatan(split_of_record)
    sidik_file = RUNS_DIR / run / 'sidik_lipatan.json'
    sidik_lama = json.loads(sidik_file.read_text())['sidik'] if sidik_file.exists() else None

    if best.exists() and sidik_lama == sidik_baru:
        print(f'    bobot sudah ada DAN isinya cocok, latihan dilewati: {best}', flush=True)
    else:
        assert not best.exists(), (
            f'fold {k}: {best} sudah ada tetapi dilatih dari daftar foto yang BERBEDA '
            f'(sidik {sidik_lama} lawan {sidik_baru}). Jangan menilai model lama dengan '
            f'data baru. Ganti RUN_PREFIX, atau hapus folder {RUNS_DIR / run}.')
        YOLO(MODEL_WEIGHTS).train(
            data=str(yml), epochs=EPOCHS_FOLD, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
            seed=SEED, project=str(RUNS_DIR), name=run, exist_ok=True, plots=False, **AUG)
        sidik_file.parent.mkdir(parents=True, exist_ok=True)
        sidik_file.write_text(json.dumps({'sidik': sidik_baru, 'n_foto': len(records),
                                          'n_uji': n['test']}))

    df = evaluasi_total(YOLO(str(best)), root)
    df['fold'] = k
    hasil_fold[k] = df
    print(f'    fold {k}: {df.benar.sum()} benar, {df.salah.sum()} salah, {df.diam.sum()} diam '
          f'dari {len(df)}', flush=True)

## 11. Angka gabungan

Setiap foto sendiri muncul **tepat sekali** di sini, diuji oleh model yang belum pernah
melihatnya. Inilah angka yang boleh dibandingkan dengan 65,7% dari notebook sebelumnya.

Dilaporkan BENAR / SALAH / DIAM, bukan satu persentase: bagi pengguna tunanetra,
nominal yang salah lebih berbahaya daripada sistem yang diam (PRD 8.9 menulis presisi
lebih diutamakan daripada recall).

In [ ]:
gabung = pd.concat(hasil_fold.values(), ignore_index=True)
n = len(gabung)
benar, salah, diam = int(gabung.benar.sum()), int(gabung.salah.sum()), int(gabung.diam.sum())

assert benar + salah + diam == n, (
    f'kategori tidak konsisten: {benar} + {salah} + {diam} != {n}. '
    'Tiap foto harus masuk TEPAT satu kategori.')
assert n == sum(1 for r in records if r['sendiri']), (
    f'{n} foto dinilai, padahal ada {sum(1 for r in records if r["sendiri"])} foto sendiri. '
    'Tiap foto sendiri harus diuji tepat sekali.')

print(f'=== k-fold gabungan atas {n} foto sendiri')
print(f'    BENAR {benar}   SALAH {salah}   DIAM {diam}   akurasi {benar / n:.1%}')
print(f'    Pembanding tanpa k-fold (06-10): BENAR 23  SALAH 6  DIAM 6  akurasi 65,7%')
print(f'    Target PRD >= 85%')

display(gabung.groupby('fold')[['benar', 'salah', 'diam']].sum())
display(gabung.groupby('total_benar')[['benar']].agg(['mean', 'count']).round(3))

print('\nJawaban SALAH (nominal keliru - paling berbahaya):')
for _, r in gabung[gabung.salah].iterrows():
    print(f"    {r.file}: benar {r.total_benar}, dijawab {r.total_prediksi} "
          f"({r.lembar_benar} lembar lawan {r.lembar_prediksi})")

gabung.to_csv(RUNS_DIR / f'{RUN_PREFIX}_kfold_totals.csv', index=False)
print(f'\nDisimpan ke {RUNS_DIR / (RUN_PREFIX + "_kfold_totals.csv")}')

## 11b. Sapuan ambang skor (TANPA melatih ulang)

`score_min` di aplikasi menentukan kapan sistem menjawab dan kapan ia diam.
Nilainya terakhir dikalibrasi 27 September pada model yang sudah dua generasi lalu,
jadi belum tentu masih cocok.

Sel di bawah **tidak melatih apa pun**: bobot tiap lipatan sudah ada di Drive, dan
dataset tiap lipatan masih ada di sesi ini. Ia hanya menilai ulang prediksi pada
beberapa ambang, sehingga hasilnya tetap bersih - tiap foto dinilai model yang
belum pernah melihatnya.

Bacalah BENAR / SALAH / DIAM, bukan akurasinya saja. Menurunkan ambang menukar
DIAM dengan BENAR **dan** SALAH sekaligus, dan bagi pengguna tunanetra nominal
yang salah lebih berbahaya daripada sistem yang diam.

In [ ]:
AMBANG_SAPU = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]

baris = []
for conf in AMBANG_SAPU:
    bagian = []
    for k in range(K):
        bobot = RUNS_DIR / f'{RUN_PREFIX}_fold{k}' / 'weights' / 'best.pt'
        akar = Path(f'/content/data_fold{k}')
        if not bobot.exists() or not (akar / 'test' / 'images').exists():
            continue
        bagian.append(evaluasi_total(YOLO(str(bobot)), akar, conf=conf))
    if not bagian:
        print('Tidak ada lipatan yang bisa dinilai. Jalankan dulu sel 10 (putaran k-fold).')
        break
    g = pd.concat(bagian, ignore_index=True)
    baris.append({'ambang': conf,
                  'BENAR': int(g.benar.sum()),
                  'SALAH': int(g.salah.sum()),
                  'DIAM': int(g.diam.sum()),
                  'akurasi': g.benar.mean()})

if baris:
    sapu = pd.DataFrame(baris).set_index('ambang')
    display(sapu.style.format({'akurasi': '{:.1%}'}))
    terbaik = sapu.BENAR.idxmax()
    aman = sapu[sapu.SALAH == sapu.SALAH.min()].index.max()
    print(f'Ambang dengan BENAR terbanyak : {terbaik}')
    print(f'Ambang dengan SALAH tersedikit: {aman}')
    print()
    print('Pilih sadar, jangan otomatis ambil yang akurasinya tertinggi:')
    print('  - BENAR terbanyak  -> aplikasi paling sering berguna')
    print('  - SALAH tersedikit -> aplikasi paling jarang menyesatkan')
    print('Kalau keduanya berbeda, PRD 8.9 memihak yang kedua (presisi di atas recall).')
    print()
    print('Setelah memilih, samakan config/settings.yaml -> vision.money.score_min,')
    print('dan ukur ulang dengan scripts/diagnosa_uang.py supaya keduanya sepadan.')
    sapu.to_csv(RUNS_DIR / f'{RUN_PREFIX}_sapuan_ambang.csv')
    print()
    print("Disimpan ke", RUNS_DIR / (RUN_PREFIX + "_sapuan_ambang.csv"))

## 12. Model terakhir, dilatih pada seluruh data

Kalau angka k-fold di atas **lebih baik** daripada 65,7%, prosedurnya layak dipakai dan
sel ini melatih model yang akan dipasang di aplikasi: seluruh foto - publik maupun sendiri -
masuk data latih, hanya validasi yang disisihkan.

Kalau angka k-fold **tidak lebih baik**, jangan jalankan sel ini. Berarti menambahkan 35
foto sendiri ke data latih tidak menolong, dan masalahnya memang kurangnya variasi data -
yang hanya bisa dijawab dengan foto baru.

**Model ini tidak bisa diukur lagi di notebook ini**, karena tidak ada data yang tersisa.
Ukur dengan foto BARU di laptop:
`python scripts\diagnosa_uang.py data\uji_sendiri --harapan-dari-nama`

In [ ]:
LATIH_MODEL_TERAKHIR = False     # ubah ke True setelah melihat angka k-fold di atas

if not LATIH_MODEL_TERAKHIR:
    print('Dilewati. Lihat angka k-fold di Bagian 11 lebih dulu, lalu ubah')
    print('LATIH_MODEL_TERAKHIR = True bila angkanya lebih baik daripada 65,7%.')
else:
    split_of_record = {}
    for g, rs in by_group.items():
        split = 'val' if g in grup_val else 'train'
        for r in rs:
            split_of_record[id(r)] = split
    # split test dibiarkan kosong: memang tidak ada lagi data yang disisihkan
    root = Path('/content/data_final')
    yml = tulis_dataset(split_of_record, root)
    n = Counter(split_of_record.values())
    print(f'latih {n["train"]}, validasi {n["val"]}, uji 0 (disengaja)')

    run = f'{RUN_PREFIX}_final'
    YOLO(MODEL_WEIGHTS).train(
        data=str(yml), epochs=EPOCHS_FINAL, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        seed=SEED, project=str(RUNS_DIR), name=run, exist_ok=True, save_period=10,
        plots=True, **AUG)
    BEST_FINAL = RUNS_DIR / run / 'weights' / 'best.pt'
    print('Bobot model terakhir:', BEST_FINAL)

## 13. Simpan dan catat keterbatasan

In [ ]:
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M')
tujuan = DATA_DIR / f'best_money_kfold_{stamp}.pt'
sumber = RUNS_DIR / f'{RUN_PREFIX}_final' / 'weights' / 'best.pt'
if sumber.exists():
    shutil.copy(sumber, tujuan)
    print('Bobot disalin ke Drive:', tujuan)
    print('Salin ke models/best_money.pt di laptop, lalu JALANKAN:')
    print(r'    python scripts\diagnosa_uang.py data\uji_sendiri --harapan-dari-nama')
    print('Perhatikan metrik TOTAL, bukan metrik KELAS.')
else:
    print('Model terakhir belum dilatih; tidak ada yang disalin.')

print("""
KETERBATASAN yang wajib masuk laporan:

1. Angka k-fold mengukur PROSEDURNYA, bukan model terakhir. Setelah model terakhir
   dilatih pada semua foto sendiri, tidak ada data tersisa untuk mengujinya.
2. Foto sendiri berasal dari sedikit latar dan sedikit kamera, dan sebarannya
   SENGAJA tidak seimbang: tambahan 23-10 diarahkan ke kelas terlemah (2000, 5000,
   100000), sedangkan 1000 hanya punya 4 kotak. k-fold tidak bisa mengarang variasi
   yang tidak ada di datanya, dan kelas bersampel kecil tetap bersampel kecil.
3. Foto banyak lembar hampir tidak ada (satu foto), jadi penghitungan jumlah lembar
   tetap klaim yang belum terbukti - baik di notebook ini maupun di laptop.
4. Metrik yang dipakai di sini adalah TOTAL (yang diucapkan ke pengguna). Angka lama
   yang memakai metrik kelas dominan lebih tinggi dan TIDAK sebanding.
""")